# Grupo 2 — validação sanitária da base de tráfego

Este notebook verifica nulos, duplicatas, datas e regularidade horária. Atenção especial à coluna `holiday`: nela, `None` pode significar semanticamente “não é feriado”, embora seja lido como nulo.

In [1]:
from pathlib import Path
import sys
import pandas as pd

RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'validacao_bases.py').is_file()), None)
if RAIZ is None:
    raise FileNotFoundError('Não foi possível localizar validacao_bases.py.')
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from validacao_bases import (
    CONFIGURACOES, calcular_sha256, carregar_base, extrair_datas,
    relatorios_como_dataframe, validar_base,
)

NOME_BASE = 'trafego'
config = CONFIGURACOES[NOME_BASE]
dados = carregar_base(NOME_BASE, RAIZ)
print(f'Base: {NOME_BASE} | formato: {dados.shape[0]:,} linhas x {dados.shape[1]} colunas')

Base: trafego | formato: 48,204 linhas x 9 colunas


In [2]:
display(dados.head())
display(dados.dtypes.rename('tipo').to_frame())

,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,NaN,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,NaN,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,NaN,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767
3,NaN,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026
4,NaN,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918


,tipo
holiday,object
temp,float64
rain_1h,float64
snow_1h,float64
clouds_all,int64
weather_main,object
weather_description,object
date_time,object
traffic_volume,int64


## Resultado consolidado

A frequência esperada é horária (`h`). Duplicatas temporais são mostradas separadamente de linhas integralmente duplicadas.

In [3]:
relatorio = validar_base(NOME_BASE, RAIZ)
display(relatorios_como_dataframe({NOME_BASE: relatorio}))
display(pd.Series(relatorio.nulos_por_coluna, name='quantidade_de_nulos').to_frame())

,nome,linhas,colunas,linhas_com_nulos,duplicatas_exatas,datas_invalidas,datas_duplicadas,ordenacao_datas,frequencia_esperada,frequencia_regular,timestamps_ausentes,timestamps_fora_da_grade,aprovada
0,trafego,48204,9,48143,17,0,7629,crescente,h,False,11976,0,False


,quantidade_de_nulos
holiday,48143


In [4]:
datas = extrair_datas(dados, config)
problemas = dados.loc[dados.duplicated(keep=False) | datas.duplicated(keep=False)].copy()
problemas.insert(0, 'data_normalizada', datas.loc[problemas.index])
print(f'Linhas com duplicidade exata ou temporal: {len(problemas):,}')
display(problemas.head(10))
print('Exemplos de horas ausentes:', relatorio.exemplos_timestamps_ausentes)
print('Ordenação encontrada:', relatorio.ordenacao_datas)

Linhas com duplicidade exata ou temporal: 13,074


,data_normalizada,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
178,2012-10-10 07:00:00,NaN,281.25,0.0,0.0,99,Rain,light rain,2012-10-10 07:00:00,6793
179,2012-10-10 07:00:00,NaN,281.25,0.0,0.0,99,Drizzle,light intensity drizzle,2012-10-10 07:00:00,6793
180,2012-10-10 08:00:00,NaN,280.10,0.0,0.0,99,Rain,light rain,2012-10-10 08:00:00,6283
181,2012-10-10 08:00:00,NaN,280.10,0.0,0.0,99,Drizzle,light intensity drizzle,2012-10-10 08:00:00,6283
182,2012-10-10 09:00:00,NaN,279.61,0.0,0.0,99,Rain,light rain,2012-10-10 09:00:00,5680
183,2012-10-10 09:00:00,NaN,279.61,0.0,0.0,99,Drizzle,light intensity drizzle,2012-10-10 09:00:00,5680
269,2012-10-14 09:00:00,NaN,282.43,0.0,0.0,57,Drizzle,light intensity drizzle,2012-10-14 09:00:00,2685
270,2012-10-14 09:00:00,NaN,282.43,0.0,0.0,57,Mist,mist,2012-10-14 09:00:00,2685
271,2012-10-14 09:00:00,NaN,282.43,0.0,0.0,57,Haze,haze,2012-10-14 09:00:00,2685
272,2012-10-14 10:00:00,NaN,282.33,0.0,0.0,57,Drizzle,light intensity drizzle,2012-10-14 10:00:00,3370


Exemplos de horas ausentes: ['2012-10-03T07:00:00', '2012-10-03T10:00:00', '2012-10-03T11:00:00', '2012-10-03T17:00:00', '2012-10-05T02:00:00']
Ordenação encontrada: crescente


## Evidência para o congelamento

O hash identifica exatamente o arquivo analisado. O congelamento completo das cinco bases deve ser feito uma única vez com `congelar_bases('dados_congelados/v1')`.

In [5]:
arquivo = RAIZ / config.caminho
print('Arquivo:', arquivo.relative_to(RAIZ))
print('SHA-256:', calcular_sha256(arquivo))
print('Conclusão:', 'APROVADA' if relatorio.aprovada else 'REQUER TRATAMENTO ANTES DA MODELAGEM')

Arquivo: grupo2\Metro_Interstate_Traffic_Volume.csv
SHA-256: 749c90d720360a4215bb15345526073c079ba4cc95e3fa558796d083f85fce9e
Conclusão: REQUER TRATAMENTO ANTES DA MODELAGEM
